# LLM Fine-Tuning for Indonesian Health Regulations

**Permenkes No. 10 Tahun 2024 - PEFT/QLoRA Fine-Tuning Pipeline**

This notebook contains the complete pipeline for:
1. **Data Preprocessing** - Extract and prepare training data from PDF
2. **Model Training** - Fine-tune LLaMA 3 8B with QLoRA
3. **Inference Demo** - Test the model with health regulation queries

**Target Environment:** Google Colab T4 GPU (16GB VRAM)

---
## Setup & Dependencies

In [1]:
# Clone repository
!git clone https://github.com/mpfordreamer/paperlesshospital-test.git
%cd paperlesshospital-test

fatal: destination path 'paperlesshospital-test' already exists and is not an empty directory.
/content/paperlesshospital-test


In [2]:
!pip install unsloth pdfplumber

In [3]:
# Install rouge score for evaluation
!pip install rouge_score

In [4]:
import re
import json
import random
from pathlib import Path

import torch
import pdfplumber
from datasets import Dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from rouge_score import rouge_scorer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/tmp/ipython-input-468160685.py:10: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, is_bfloat16_supported


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


---
## Configuration

In [5]:
# Paths
PDF_PATH = Path("data/raw/permenkes-no-10-tahun-2024.pdf")
DATASET_PATH = Path("data/dataset.jsonl")
OUTPUT_DIR = Path("outputs")

# Model
BASE_MODEL = "unsloth/llama-3-8b-bnb-4bit"

# LoRA Configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training Hyperparameters
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
MAX_STEPS = 100
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512

---
# Phase 1: Data Preprocessing

Extract text from PDF, parse articles (Pasal), and generate instruction-tuning dataset.

### 1.1 PDF Text Extraction

In [6]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract raw text from PDF."""
    full_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
    return "\n\n".join(full_text)

raw_text = extract_text_from_pdf(PDF_PATH)
print(f"Extracted {len(raw_text):,} characters")

Extracted 11,850 characters


### 1.2 Article Pasal Parsing & Cleaning

In [7]:
def parse_articles(text: str) -> list:
    """Parse articles with anti-reference logic and noise cleaning."""

    # Remove preamble
    if "MEMUTUSKAN" in text:
        split_text = text.split("MEMUTUSKAN", 1)[1]
    else:
        split_text = text

    # Find all "Pasal X" candidates
    candidate_pattern = r'(?i)(Pasal\s+(\d+))'
    matches = list(re.finditer(candidate_pattern, split_text))

    valid_headers = []

    # Filter: separate real headers from references
    for match in matches:
        start_idx = match.start()
        preceding_text = split_text[max(0, start_idx-30):start_idx].lower()
        preceding_clean = re.sub(r'\s+', ' ', preceding_text).strip()

        reference_indicators = ['dalam', 'pada', 'sesuai', 'angka', 'huruf', 'ayat', 'tentang', 'sebagaimana']
        is_reference = any(preceding_clean.endswith(ind) for ind in reference_indicators)

        if not is_reference:
            valid_headers.append(match)

    # Extract content from valid headers
    articles = []
    for i, match in enumerate(valid_headers):
        pasal_num = match.group(2)
        start_content = match.end()
        end_content = valid_headers[i+1].start() if i < len(valid_headers) - 1 else len(split_text)

        content_raw = split_text[start_content:end_content]

        # CLEANING
        content_clean = re.sub(r'\s+', ' ', content_raw).strip()

        # Remove page numbers (multiple formats: -3-, - 3 -, --3--)
        content_clean = re.sub(r'-+\s*\d+\s*-+', '', content_clean)

        # Remove PDF artifacts (extended list)
        content_clean = re.sub(r'[ŒДѼЖÐÑ®™©°±²³¹\x00-\x1f]', '', content_clean)

        # Remove "NOMOR Ж" or similar patterns
        content_clean = re.sub(r'NOMOR\s*[ŒДѼЖÐÑ\W]*', 'NOMOR ', content_clean)

        # Remove leading symbols
        content_clean = re.sub(r'^[.:\-\s]+', '', content_clean)

        # Final whitespace cleanup
        content_clean = re.sub(r'\s+', ' ', content_clean).strip()

        if len(content_clean) > 10:
            articles.append({"number": pasal_num, "content": content_clean})

    return articles

In [8]:
# Extract and parse
raw_text = extract_text_from_pdf(PDF_PATH)
articles = parse_articles(raw_text)

# Verify results
print(f"Found {len(articles)} articles.")

# Check Pasal 5 content is correct (should be "Tugas Anggota", not preamble)
for art in articles:
    if art['number'] == '5':
        print(f"\n[CHECK PASAL 5]: {art['content'][:200]}...")

Found 11 articles.

[CHECK PASAL 5]: (1) Anggota JDIH Kemenkes sebagaimana dimaksud dalam Pasal 3 ayat (2) huruf b mempunyai tugas mengelola Dokumen Hukum dan Informasi Hukum yang diterbitkan oleh unit kerja di lingkungan Eselon I masing...


### 1.3 Generate Instruction-Tuning Dataset

In [9]:
def generate_qa_pairs(articles: list, full_text: str, min_examples: int = 50) -> list:
    """Generate diverse Q&A pairs with Answer-First format."""

    qa_pairs = []

    # Generic templates (3 variations)
    generic_templates = [
        # Formal
        ("Jelaskan isi {p} dalam Permenkes No 10 Tahun 2024.",
         "Jawaban: Berdasarkan regulasi, {p} mengatur hal berikut.\n\nIsi Pasal: {c}"),

        # Direct
        ("Apa yang tertulis dalam {p}?",
         "Jawaban: Berikut adalah isi lengkap {p}.\n\nKutipan: {c}"),

        # Casual
        ("Bisa sebutkan isi {p}?",
         "Jawaban: Tentu, {p} berbunyi sebagai berikut: {c}")
    ]

    for art in articles:
        pasal = f"Pasal {art['number']}"
        content = art['content']
        content_lower = content.lower()

        # Apply generic templates
        for q_tmpl, a_tmpl in generic_templates:
            qa_pairs.append({
                "instruction": q_tmpl.format(p=pasal),
                "input": f"Konteks: Permenkes No. 10 Tahun 2024, {pasal}",
                "output": a_tmpl.format(p=pasal, c=content)
            })

        # Specific: Obligation/Prohibition
        if any(w in content_lower for w in ['wajib', 'harus', 'dilarang', 'sanksi', 'memerintahkan']):
            qa_pairs.append({
                "instruction": f"Apa kewajiban atau perintah yang diatur dalam {pasal}?",
                "input": f"Konteks: Permenkes No. 10 Tahun 2024, {pasal}",
                "output": f"Jawaban: {pasal} memuat kewajiban/perintah yang harus dipatuhi.\n\nRincian: {content}"
            })

        # Specific: Definition
        if art['number'] == '1' or 'dimaksud dengan' in content_lower:
            qa_pairs.append({
                "instruction": f"Apa definisi yang dijelaskan pada {pasal}?",
                "input": f"Konteks: Permenkes No. 10 Tahun 2024, {pasal}",
                "output": f"Jawaban: {pasal} menjelaskan definisi terminologi dalam peraturan ini.\n\nDefinisi Lengkap: {content}"
            })

        # Specific: Task/Authority
        if any(w in content_lower for w in ['tugas', 'wewenang', 'fungsi', 'bertanggung jawab']):
            qa_pairs.append({
                "instruction": f"Sebutkan tugas atau fungsi yang ada di {pasal}.",
                "input": f"Konteks: Permenkes No. 10 Tahun 2024, {pasal}",
                "output": f"Jawaban: {pasal} merinci tugas, fungsi, atau wewenang terkait.\n\nUraian: {content}"
            })

        # Data augmentation: Scenario question
        qa_pairs.append({
            "instruction": f"Jika saya ingin mencari informasi tentang topik di {pasal}, apa isinya?",
            "input": f"Konteks: Permenkes No. 10 Tahun 2024",
            "output": f"Anda dapat merujuk pada {pasal}. Isinya adalah: {content}"
        })

    # Shuffle to avoid order bias
    random.shuffle(qa_pairs)

    return qa_pairs


### 1.4 Save Dataset

In [10]:
articles = parse_articles(raw_text)

qa_pairs = generate_qa_pairs(articles, raw_text, min_examples=100)
print(f"Generated {len(qa_pairs)} Q&A pairs")

with open(DATASET_PATH, 'w', encoding='utf-8') as f:
    for item in qa_pairs:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Dataset saved to {DATASET_PATH}")
print(f"\nSample entry:")
print(json.dumps(qa_pairs[0], indent=2, ensure_ascii=False))

Generated 50 Q&A pairs
Dataset saved to data/dataset.jsonl

Sample entry:
{
  "instruction": "Bisa sebutkan isi Pasal 11?",
  "input": "Konteks: Permenkes No. 10 Tahun 2024, Pasal 11",
  "output": "Jawaban: Tentu, Pasal 11 berbunyi sebagai berikut: Peraturan Menteri ini mulai berlaku pada tanggal diundangkan. Agar setiap orang mengetahuinya, memerintahkan pengundangan Peraturan Menteri ini dengan penempatannya dalam Berita Negara Republik Indonesia. Ditetapkan di Jakarta pada tanggal 2 Agustus 2024 MENTERI KESEHATAN REPUBLIK INDONESIA, BUDI G. SADIKIN Diundangkan di Jakarta pada tanggal PLT. DIREKTUR JENDERAL PERATURAN PERUNDANG-UNDANGAN KEMENTERIAN HUKUM DAN HAK ASASI MANUSIA REPUBLIK INDONESIA, ASEP N. MULYANA BERITA NEGARA REPUBLIK INDONESIA TAHUN 2024 NOMOR"
}


---
# Phase 2: Model Training

Fine-tune LLaMA 3 8B with QLoRA on T4 GPU.

### 2.1 Load Dataset

In [11]:
def load_dataset_from_jsonl(path: Path) -> Dataset:
    """Load JSONL dataset and format for training."""
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return Dataset.from_list(data)


def format_prompt(example: dict) -> str:
    """Format example into instruction-following prompt."""
    return f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""


dataset = load_dataset_from_jsonl(DATASET_PATH)
dataset = dataset.map(lambda x: {"text": format_prompt(x)})
print(f"Loaded {len(dataset)} training examples")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Loaded 50 training examples


### 2.2 Load Base Model (4-bit Quantized)

In [12]:
# Using Unsloth's FastLanguageModel for optimized loading
max_seq_length = MAX_SEQ_LENGTH # Max sequence length for model
dtype = None # None for auto detection. Float16 for Tesla T4, V100, BFC for Ampere+
load_in_4bit = True # Use 4bit quantization to save memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL, # Choose any of the following models to train for free on Google Colab T4 GPUs
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Unsloth's tokenizer setup
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {BASE_MODEL}")

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded: unsloth/llama-3-8b-bnb-4bit


### 2.3 Configure LoRA Adapter

In [13]:
# Configure LoRA adapter with Unsloth's optimized method
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = LORA_TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_SEQ_LENGTH,
)
model.print_trainable_parameters()

Unsloth 2026.1.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [ ]:
# Post-Training Inference Test before fine-tuning
print("--- TESTING MODEL (POST-TRAINING) ---")

# Set model to inference mode
FastLanguageModel.for_inference(model)

def test_model(question):
    """Test the base model before fine-tuning."""
    prompt = f"""### Instruction:
{question}

### Input:
Konteks: Permenkes No. 10 Tahun 2024

### Response:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.5)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

# Test with a key question
test_q = "Apa definisi JDIH Kemenkes menurut peraturan ini? apakah kamu tahu pasal nya berapa?"
print(f"Question: {test_q}")
print(f"Base Model Response:\n{test_model(test_q)}")
print("\n" + "="*60 + "\n")

--- TESTING BASE MODEL (PRE-TRAINING) ---
Question: Apa definisi JDIH Kemenkes menurut peraturan ini? apakah kamu tahu pasal nya berapa?
Base Model Response:
Jawaban: Tentu, Pasal 2

Kutipan: JDIH Kemenkes merupakan wadah pengelolaan dokumentasi dan Informasi Hukum Kementerian Kesehatan yang berfungsi sebagai sarana pendukung kegiatan pengelolaan hukum di Kementerian Kesehatan. JDIH Kemenkes sebagaimana dimaksud dalam Pasal 1 ayat (2) huruf b memuat semua dokumentasi dan Informasi Hukum yang diterbitkan selain dari unit kerja Eselon I




**Base Model Response (Sebelum Fine-Tuning):**
> Jawaban: Tentu, Pasal 2  
> Kutipan: JDIH Kemenkes merupakan wadah pengelolaan dokumentasi dan Informasi Hukum Kementerian Kesehatan yang berfungsi sebagai sarana pendukung kegiatan pengelolaan hukum di Kementerian Kesehatan. JDIH Kemenkes sebagaimana dimaksud dalam Pasal 1 ayat (2) huruf b memuat semua dokumentasi dan Informasi Hukum yang diterbitkan selain dari unit kerja Eselon I

Model base mengikuti format instruksi namun konten belum ter-grounding ke dokumen spesifik.

### 2.4 Training

In [14]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_steps=20,
    warmup_steps=5,
    optim="paged_adamw_8bit",
    save_total_limit=2,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

print("Starting training...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 15 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.967600
20,0.205100
30,0.047600
40,0.034600
50,0.024200
60,0.019200
70,0.018200
80,0.014600
90,0.013300
100,0.014400


TrainOutput(global_step=100, training_loss=0.13588491663336755, metrics={'train_runtime': 812.8906, 'train_samples_per_second': 0.984, 'train_steps_per_second': 0.123, 'total_flos': 1.0476305164296192e+16, 'train_loss': 0.13588491663336755, 'epoch': 14.32})

### 2.5 Save LoRA Adapter

In [15]:
model.save_pretrained(OUTPUT_DIR / "lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR / "lora_adapter")
print(f"Adapter saved to {OUTPUT_DIR / 'lora_adapter'}")

Adapter saved to outputs/lora_adapter


---
# Phase 3: Inference Demo

Test the fine-tuned model with health regulation queries.

### 3.1 Load Fine-tuned Model

In [16]:
# Model is already loaded from training
FastLanguageModel.for_inference(model)
print("Fine-tuned model ready for inference")


Fine-tuned model ready for inference


### 3.2 Inference Function

In [17]:
def generate_answer(question: str, context: str = "") -> str:
    """Generate answer with article citation."""
    prompt = f"""### Instruction:
{question}

### Input:
{context if context else 'Konteks: Permenkes No. 10 Tahun 2024'}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()


# Test it
test_q = "Apa tugas Pusat JDIH Kemenkes?"
print(f"Q: {test_q}")
print(f"A: {generate_answer(test_q)}")

Q: Apa tugas Pusat JDIH Kemenkes?
A: Jawaban: Berdasarkan regulasi, tugas Pusat JDIH Kemenkes adalah: (1) merumuskan kebijakan pembinaan, pengelolaan, dan pengembangan JDIH Kemenkes; (2) memberikan rujukan dokumentasi dan Informasi Hukum Kementerian Kesehatan; dan (3) mengelola Dokumen Hukum dan Informasi Hukum yang diterbitkan selain dari unit


### 3.3 Test Queries

In [18]:
test_questions = [
    "Apa yang diatur dalam Pasal 1?",
    "Jelaskan tentang jaringan dokumentasi dan informasi hukum.",
    "Apa kewajiban unit kerja dalam pengelolaan dokumen hukum?",
]

print("=" * 60)
print("INFERENCE DEMO")
print("=" * 60)

for q in test_questions:
    print(f"\nQ: {q}")
    answer = generate_answer(q)
    print(f"A: {answer}")
    print("-" * 60)

INFERENCE DEMO

Q: Apa yang diatur dalam Pasal 1?
A: Jawaban: Pasal 1 memuat definisi terminologi dalam peraturan ini.

Definisi Lengkap: Dalam Peraturan Menteri ini yang dimaksud dengan: 1. Jaringan Dokumentasi dan Informasi Hukum Nasional yang selanjutnya disingkat JDIHN adalah wadah pendayagunaan bersama atas dokumen hukum secara tertib, terpadu, dan berkesinambungan, serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah, dan cepat. 2. Jaringan Dokumentasi dan Informasi Hukum di lingkungan Kementerian Kesehatan yang selanjutnya disebut JDIH Kemenkes adalah suatu sistem pengelolaan dan pendayagunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu dan berkesinambungan serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah dan cepat. 3. Dokumen Hukum adalah produk
------------------------------------------------------------

Q: Jelaskan tentang jaringan dokumentasi dan informasi hu

## 3.4 Evaluation Rouge Score

In [19]:
# Initialize Scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Test Data - Permenkes No. 10 Tahun 2024
test_data = [
    # Definition (Pasal 1)
    {
        "question": "Apa definisi JDIH Kemenkes menurut peraturan ini?",
        "ground_truth": "Berdasarkan Pasal 1 angka 2, JDIH Kemenkes adalah suatu sistem pengelolaan dan pendayagunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu dan berkesinambungan serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah dan cepat."
    },

    # Objectives (Pasal 2)
    {
        "question": "Apa tujuan dari JDIH Kemenkes?",
        "ground_truth": "Sesuai Pasal 2, tujuannya adalah: a. mewujudkan tertib administrasi dokumen hukum; b. melakukan penataan dokumentasi melalui teknologi informasi; dan c. menyediakan informasi hukum bidang kesehatan secara lengkap, akurat, mudah, dan cepat."
    },

    # Organization Members (Pasal 3)
    {
        "question": "Sebutkan siapa saja yang termasuk Anggota JDIH Kemenkes.",
        "ground_truth": "Berdasarkan Pasal 3 ayat (4), Anggota JDIH Kemenkes terdiri atas Sekretariat dari: Direktorat Jenderal Kesehatan Masyarakat, Pencegahan dan Pengendalian Penyakit, Pelayanan Kesehatan, Kefarmasian dan Alat Kesehatan, Tenaga Kesehatan, Inspektorat Jenderal, Badan Kebijakan dan Pembangunan Kesehatan, serta Sekretariat Konsil."
    },

    # Center Tasks (Pasal 4)
    {
        "question": "Apa tugas Pusat JDIH Kemenkes?",
        "ground_truth": "Sesuai Pasal 4 ayat (1), tugasnya adalah merumuskan kebijakan pembinaan, memberikan rujukan dokumentasi, dan mengelola Dokumen Hukum yang diterbitkan selain dari unit kerja Eselon I anggota JDIH Kemenkes."
    },

    # Member Tasks (Pasal 5)
    {
        "question": "Apa tugas dari Anggota JDIH Kemenkes?",
        "ground_truth": "Sesuai Pasal 5 ayat (1), Anggota JDIH bertugas mengelola Dokumen Hukum dan Informasi Hukum yang diterbitkan oleh unit kerja di lingkungan Eselon I masing-masing."
    },

    # Management Methods (Pasal 6)
    {
        "question": "Melalui apa pengelolaan dokumentasi dan informasi hukum dilakukan?",
        "ground_truth": "Berdasarkan Pasal 6 ayat (5), pengelolaan dilakukan melalui website jdih.kemkes.go.id yang terhubung dengan website Kementerian Kesehatan dan terintegrasi dengan Pusat JDIHN."
    },

    # Technical Team (Pasal 7)
    {
        "question": "Siapa saja unsur yang tergabung dalam Tim Teknis JDIH Kemenkes?",
        "ground_truth": "Menurut Pasal 7 ayat (2), Tim teknis berasal dari unsur pusat JDIH Kemenkes, anggota JDIH Kemenkes, dan Pusat Data dan Teknologi Informasi."
    },

    # Document Types (Pasal 8)
    {
        "question": "Apa saja jenis Dokumen Hukum yang dikelola dalam JDIH Kemenkes?",
        "ground_truth": "Sesuai Pasal 8 ayat (1), dokumen hukum meliputi Peraturan Perundang-undangan, produk hukum lain, monografi, artikel hukum, dan putusan/yurisprudensi."
    },

    # Monitoring (Pasal 9)
    {
        "question": "Berapa kali monitoring dan evaluasi dilaksanakan?",
        "ground_truth": "Berdasarkan Pasal 9 ayat (2), monitoring dan evaluasi dilaksanakan paling sedikit 1 (satu) kali dalam setahun."
    },

    # Funding (Pasal 10)
    {
        "question": "Dari mana sumber pendanaan pengelolaan JDIH Kemenkes?",
        "ground_truth": "Sesuai Pasal 10, pendanaan bersumber pada anggaran pendapatan dan belanja negara (APBN) dan/atau sumber lain yang sah sesuai ketentuan peraturan perundang-undangan."
    },

    # Closing/Effectivity (Pasal 11)
    {
        "question": "Kapan Peraturan Menteri ini mulai berlaku?",
        "ground_truth": "Berdasarkan Pasal 11, Peraturan Menteri ini mulai berlaku pada tanggal diundangkan (7 Agustus 2024)."
    }
]

print("=" * 60)
print("MODEL EVALUATION WITH ROUGE SCORES")
print("=" * 60)

# Track average scores
total_rouge1, total_rouge2, total_rougeL = 0, 0, 0

for i, item in enumerate(test_data, 1):
    q = item['question']
    ground_truth = item['ground_truth']

    # Generate answer
    print(f"\n[{i}/{len(test_data)}] Q: {q}")
    model_answer = generate_answer(q)
    print(f"Model: {model_answer}")
    print(f"Truth: {ground_truth}")

    # Calculate scores
    scores = scorer.score(ground_truth, model_answer)
    rouge1 = scores['rouge1'].fmeasure
    rouge2 = scores['rouge2'].fmeasure
    rougeL = scores['rougeL'].fmeasure

    total_rouge1 += rouge1
    total_rouge2 += rouge2
    total_rougeL += rougeL

    print(f"ROUGE-1: {rouge1:.4f} | ROUGE-2: {rouge2:.4f} | ROUGE-L: {rougeL:.4f}")
    print("-" * 60)

# Summary Statistics
n = len(test_data)
print("\n" + "=" * 60)
print("AVERAGE SCORES")
print("=" * 60)
print(f"ROUGE-1 (Unigram): {total_rouge1/n:.4f}")
print(f"ROUGE-2 (Bigram):  {total_rouge2/n:.4f}")
print(f"ROUGE-L (LCS):     {total_rougeL/n:.4f}")


MODEL EVALUATION WITH ROUGE SCORES

[1/11] Q: Apa definisi JDIH Kemenkes menurut peraturan ini?
Model: Jawaban: Berdasarkan regulasi, definisi JDIH Kementerian Kesehatan adalah wadah pengelolaan dokumentasi dan Informasi Hukum Kementerian Kesehatan yang selaras dengan kebijakan pengelolaan JDIHN.

Rincian: (1) Dalam rangka penyelenggaraan pengelolaan JDIH Kemenkes dibentuk organisasi pengelola JDIH Kemenkes. (2) Organisasi pengelola JDIH Kemenkes sebagaimana dimaksud pada ayat (1) terdiri atas: a. pusat JDIH Kemenkes; dan b. anggota JDIH Kemenkes. (3) Pusat JDIH Kemenkes sebagaimana dimaksud pada ayat (2) huruf a dilaksanakan oleh Biro Hukum. (4) Anggota JDIH Kemenkes sebagaimana dimaksud pada ayat (2) huruf b terdiri atas: a. Sekretariat Direktorat Jenderal Kesehatan Masyarakat; b. Sekretariat Direktorat
Truth: Berdasarkan Pasal 1 angka 2, JDIH Kemenkes adalah suatu sistem pengelolaan dan pendayagunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpad

## 3.5 Interactive Demo

In [20]:
# Interactive query (uncomment to use)
user_question = input("Masukkan pertanyaan tentang Permenkes: ")
print(f"\n{generate_answer(user_question)}")

Masukkan pertanyaan tentang Permenkes: berapa pasal di permenkes ini?

Jawaban: Terdapat 11 pasal dalam Peraturan Menteri ini.

Rincian: Pasal 1 JDIH Kemenkes merupakan suatu sistem pengelolaan dokument Hukum dan Informasi Hukum yang berlangsung secara tertib, terpadu, dan berkesinambungan; serta telah didukung oleh teknologi informasi dan komunikasi. Pasal 2 (1) Dalam rangka penyelenggaraan JD I H Kemenkes sebagaimana dimaksud dalam Pasal 1 dipusatkan pada Biro Hukum. (2) Pusat JDIH Kemenkes sebagaimana dimaksud pada ayat (1) memiliki tugas: a. merumuskan kebijakan pembinaan, pengelolaan, dan pengembangan JDIH Kemenkes; b. memberikan rujukan dokument hukum dan informasi hukum kementerian; dan c. mengelola Dokumen Hukum dan Informasi Hukum yang diterbitkan selain dari unit kerja Eselon I anggota JDIH Kemenkes. (3) Untuk melaks


---
## Summary

| Phase | Status | Output |
|-------|--------|--------|
| Data Preprocessing | ✅ Complete | `data/dataset.jsonl` (50 Q&A pairs) |
| Model Training | ✅ Complete | `outputs/lora_adapter/` |
| Inference Demo | ✅ Complete | Article citations working |
| Evaluation | ✅ Complete | ROUGE scores calculated |

### Training Results
| Metric | Value |
|--------|-------|
| Training Steps | 100 |
| Training Time | ~13 min |
| Final Loss | 0.0144 |
| Epochs | ~14 |

### Evaluation Scores (ROUGE)
| Metric | Score |
|--------|-------|
| ROUGE-1 (Unigram) | 0.2402 |
| ROUGE-2 (Bigram) | 0.1421 |
| ROUGE-L (LCS) | 0.2009 |

### Technical Requirements Met
- ✅ PDF extraction and JSONL generation (50 examples)
- ✅ 4-bit quantization with BitsAndBytes
- ✅ QLoRA fine-tuning optimized for T4 GPU
- ✅ Unsloth for 2x faster training
- ✅ Inference with Pasal citations
- ✅ ROUGE score evaluation